# 6. Metadata Filtering

**RAG Pipeline Series — Notebook 6**

Notebook 5 indexed `rag.pdf`'s chapter-tagged chunks into Chroma and searched them with pure similarity. But similarity search only knows about *meaning* — it has no idea a chunk came from Chapter 5 versus Chapter 12. Every chunk we indexed also carries `chapter_num` / `chapter_title` metadata, and Chroma lets us **filter** on that metadata before it ranks results, so a query can be scoped to "only this chapter," "any chapter but this one," or "chapters 9 through 12."

In this notebook we will:
1. Rebuild the same Chroma store from notebook 5.
2. See why similarity search alone can't scope a query to part of the document.
3. Filter with equality (`$eq`), exclusion (`$ne`), and set membership (`$in` / `$nin`).
4. Use numeric range filters (`$gt`, `$gte`, `$lt`, `$lte`) — and why they need a numeric metadata field, not the zero-padded chapter string.
5. Combine conditions with `$and` / `$or`, and hit Chroma's "one operator" rule head-on.
6. Pass filters through the `.as_retriever()` interface from notebook 5.
7. See the trade-off filtering makes: it can guarantee scope, but it can also filter the right answer out entirely.

## Setup

In [ ]:
%pip install -q -U langchain langchain-community pypdf sentence-transformers langchain-huggingface langchain-chroma chromadb pandas

## 1. Recap: rebuilding the Chroma store from notebook 5

Same loading, chapter-aware chunking, embedding, and indexing steps as notebook 5 — copied here so this notebook runs standalone. One addition: alongside the existing `chapter_num` string (`"01"`, `"02"`, ...) we also store `chapter_num_int` (`1`, `2`, ...). We'll see in section 5 why the numeric copy is necessary.

In [ ]:
# Only runs inside Colab. Opens a file picker; select rag.pdf.
# Safe to skip this cell if you're running locally and already have the file on disk.
from rag_utils import maybe_colab_upload

maybe_colab_upload()

In [1]:
from rag_utils import resolve_pdf_path, load_clean_text

# Prefer the Colab upload location; fall back to this repo's dataset/ folder
# when running locally (rag-notebooks/ and dataset/ are sibling folders).
PDF_PATH = resolve_pdf_path()
pages, full_text = load_clean_text(PDF_PATH)

print(f"Loaded {len(pages)} pages from {PDF_PATH}")

d:\youtube\TheAIGuy\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 58 pages from ..\dataset\rag.pdf


In [2]:
from rag_utils import DEFAULT_CHUNK_OVERLAP, DEFAULT_CHUNK_SIZE, chunk_chapters, split_into_chapters

CHUNK_SIZE, CHUNK_OVERLAP = DEFAULT_CHUNK_SIZE, DEFAULT_CHUNK_OVERLAP
chapters = split_into_chapters(full_text)

# numeric_chapter_num=True also attaches chapter_num_int (1, 2, ...) alongside the
# zero-padded chapter_num string ("01", "02", ...) - range filters need it, see section 5.
chunks = chunk_chapters(chapters, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP, numeric_chapter_num=True)

print(f"{len(chapters)} chapters -> {len(chunks)} chapter-tagged chunks")
print(chunks[0].metadata)

15 chapters -> 181 chapter-tagged chunks
{'chapter_num': '01', 'chapter_title': 'Introduction to RAG', 'chapter_num_int': 1}


In [3]:
from rag_utils import DEFAULT_EMBEDDING_MODEL, get_embedder

EMBEDDING_MODEL = DEFAULT_EMBEDDING_MODEL  # same tiny model as notebooks 4 and 5
embeddings = get_embedder(EMBEDDING_MODEL)

Loading weights: 100%|██████████| 55/55 [00:01<00:00, 47.55it/s] 


In [4]:
from rag_utils import build_chroma_store

vectorstore = build_chroma_store(chunks, embeddings=embeddings, collection_name="rag_pdf_chapters")
print(f"Indexed {vectorstore._collection.count()} chunks into Chroma")

Indexed 181 chunks into Chroma


## 2. Why filter by metadata?

A similarity search ranks every chunk in the collection by embedding distance — it has no concept of "stay inside Chapter 5." If two chapters both discuss related ideas (e.g. embeddings in Chapter 4 and indexing in Chapter 5), a broad query pulls chunks from both, in whatever order the embeddings happen to rank them. Watch what an unfiltered search over a cross-cutting query returns:

In [5]:
scope_query = "how does a vector database index embeddings for fast search?"

print("Unfiltered:")
for doc, score in vectorstore.similarity_search_with_score(scope_query, k=6):
    print(f"  score={score:.4f}  chapter={doc.metadata['chapter_num']} ({doc.metadata['chapter_title']})")

Unfiltered:
  score=0.6348  chapter=05 (Vector Databases & Indexing)
  score=0.7121  chapter=05 (Vector Databases & Indexing)
  score=0.8258  chapter=05 (Vector Databases & Indexing)
  score=0.8303  chapter=05 (Vector Databases & Indexing)
  score=0.8750  chapter=02 (Evolution of Retrieval)
  score=0.9369  chapter=13 (RAG with Tools)


## 3. Equality filters — restrict to one chapter

`similarity_search` and `similarity_search_with_score` both take a `filter` argument. Pass a plain `{field: value}` dict and Chroma treats it as shorthand for `{field: {"$eq": value}}` — only chunks whose metadata matches exactly are eligible, and the similarity ranking runs *within* that subset.

In [6]:
print("Filtered to Chapter 05 (Vector Databases & Indexing):")
for doc, score in vectorstore.similarity_search_with_score(scope_query, k=6, filter={"chapter_num": "05"}):
    print(f"  score={score:.4f}  chapter={doc.metadata['chapter_num']} ({doc.metadata['chapter_title']})")

Filtered to Chapter 05 (Vector Databases & Indexing):
  score=0.6348  chapter=05 (Vector Databases & Indexing)
  score=0.7121  chapter=05 (Vector Databases & Indexing)
  score=0.8258  chapter=05 (Vector Databases & Indexing)
  score=0.8303  chapter=05 (Vector Databases & Indexing)
  score=0.9880  chapter=05 (Vector Databases & Indexing)
  score=1.0441  chapter=05 (Vector Databases & Indexing)


## 4. Excluding and matching sets — `$ne`, `$in`, `$nin`

Beyond exact equality, Chroma supports comparison operators as the value of a metadata field:
- `$ne` — not equal to
- `$in` — value is one of a list
- `$nin` — value is none of a list

In [7]:
print("Excluding Chapter 05 ($ne) - what else in the book touches this topic:")
for doc, score in vectorstore.similarity_search_with_score(scope_query, k=5, filter={"chapter_num": {"$ne": "05"}}):
    print(f"  score={score:.4f}  chapter={doc.metadata['chapter_num']} ({doc.metadata['chapter_title']})")

foundations_query = "why do we augment a language model with retrieval instead of just fine-tuning it?"
print("\nRestricted to foundational chapters 01-03 ($in):")
for doc, score in vectorstore.similarity_search_with_score(foundations_query, k=5, filter={"chapter_num": {"$in": ["01", "02", "03"]}}):
    print(f"  score={score:.4f}  chapter={doc.metadata['chapter_num']} ({doc.metadata['chapter_title']})")

print("\nDropping the tooling/agentic chapters 13-15 ($nin):")
for doc, score in vectorstore.similarity_search_with_score(foundations_query, k=5, filter={"chapter_num": {"$nin": ["13", "14", "15"]}}):
    print(f"  score={score:.4f}  chapter={doc.metadata['chapter_num']} ({doc.metadata['chapter_title']})")

Excluding Chapter 05 ($ne) - what else in the book touches this topic:
  score=0.8750  chapter=02 (Evolution of Retrieval)
  score=0.9369  chapter=13 (RAG with Tools)
  score=0.9499  chapter=06 (Retrieval Techniques)
  score=1.0009  chapter=15 (Production Best Practices)
  score=1.0020  chapter=06 (Retrieval Techniques)

Restricted to foundational chapters 01-03 ($in):
  score=0.9512  chapter=01 (Introduction to RAG)
  score=0.9696  chapter=02 (Evolution of Retrieval)
  score=0.9703  chapter=02 (Evolution of Retrieval)
  score=1.0154  chapter=02 (Evolution of Retrieval)
  score=1.0343  chapter=02 (Evolution of Retrieval)

Dropping the tooling/agentic chapters 13-15 ($nin):
  score=0.7693  chapter=09 (Augmentation)
  score=0.8953  chapter=08 (Re-ranking)
  score=0.9512  chapter=01 (Introduction to RAG)
  score=0.9568  chapter=12 (RAG vs Fine-tuning)
  score=0.9696  chapter=02 (Evolution of Retrieval)


## 5. Numeric range filters — `$gt`, `$gte`, `$lt`, `$lte`

Range operators need a genuinely numeric field. Chroma validates operand types strictly: even though `chapter_num` is zero-padded (`"01"`...`"15"`, so string order happens to match numeric order here), passing a string to `$gte` fails outright.

In [8]:
try:
    vectorstore.similarity_search(scope_query, k=5, filter={"chapter_num": {"$gte": "09"}})
except ValueError as e:
    print("String field + range operator ->", e)

print("\nSame filter on the numeric chapter_num_int field instead:")
for doc, score in vectorstore.similarity_search_with_score(
    "how do we get an agent to call external tools safely in production?",
    k=5,
    filter={"chapter_num_int": {"$gte": 9}},
):
    print(f"  score={score:.4f}  chapter={doc.metadata['chapter_num']} ({doc.metadata['chapter_title']})")

String field + range operator -> Expected operand value to be an int or a float for operator $gte, got 09 in query.

Same filter on the numeric chapter_num_int field instead:
  score=1.2291  chapter=14 (Agentic RAG)
  score=1.2856  chapter=14 (Agentic RAG)
  score=1.3358  chapter=14 (Agentic RAG)
  score=1.3426  chapter=13 (RAG with Tools)
  score=1.3579  chapter=14 (Agentic RAG)


## 6. Compound filters — `$and` / `$or`, and the "one operator" rule

Combine multiple conditions with `$and` / `$or`, each taking a list of filter clauses. Chroma is strict about this: a `where` clause is only allowed **one top-level operator**. Passing two plain fields side by side (or an empty `{}`) isn't treated as an implicit AND — it raises a `ValueError`, so multi-condition filters must be wrapped explicitly.

In [9]:
print("Bounded range with $and - chapters 9 through 12 (Augmentation -> RAG vs Fine-tuning):")
for doc, score in vectorstore.similarity_search_with_score(
    "how do we augment a prompt with retrieved context before generation?",
    k=5,
    filter={"$and": [{"chapter_num_int": {"$gte": 9}}, {"chapter_num_int": {"$lte": 12}}]},
):
    print(f"  score={score:.4f}  chapter={doc.metadata['chapter_num']} ({doc.metadata['chapter_title']})")

print("\nBookends with $or - only Chapter 01 or Chapter 15:")
for doc, score in vectorstore.similarity_search_with_score(
    "what does it take to run this system reliably?",
    k=5,
    filter={"$or": [{"chapter_num": "01"}, {"chapter_num": "15"}]},
):
    print(f"  score={score:.4f}  chapter={doc.metadata['chapter_num']} ({doc.metadata['chapter_title']})")

print("\nTwo plain fields without $and - fails:")
try:
    vectorstore.similarity_search(
        scope_query, k=5,
        filter={"chapter_num_int": 5, "chapter_title": "Vector Databases & Indexing"},
    )
except ValueError as e:
    print(" ->", e)

Bounded range with $and - chapters 9 through 12 (Augmentation -> RAG vs Fine-tuning):
  score=0.6485  chapter=09 (Augmentation)
  score=1.0451  chapter=10 (Generation)
  score=1.0730  chapter=11 (Generation Evaluation)
  score=1.1247  chapter=09 (Augmentation)
  score=1.1388  chapter=09 (Augmentation)

Bookends with $or - only Chapter 01 or Chapter 15:
  score=1.5571  chapter=01 (Introduction to RAG)
  score=1.5715  chapter=15 (Production Best Practices)
  score=1.6068  chapter=01 (Introduction to RAG)
  score=1.6429  chapter=15 (Production Best Practices)
  score=1.6472  chapter=01 (Introduction to RAG)

Two plain fields without $and - fails:
 -> Expected where to have exactly one operator, got {'chapter_num_int': 5, 'chapter_title': 'Vector Databases & Indexing'} in query.


## 7. Filtering through the retriever interface

`.as_retriever()` (introduced in notebook 5) accepts the same `filter` inside `search_kwargs`, so a scoped search drops straight into an LCEL chain just like an unscoped one.

In [10]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3, "filter": {"chapter_num_int": {"$gte": 13}}})
retrieved = retriever.invoke("how does an agent decide which tool to call?")
[d.metadata for d in retrieved]

[{'chapter_num': '14', 'chapter_num_int': 14, 'chapter_title': 'Agentic RAG'},
 {'chapter_num_int': 14, 'chapter_num': '14', 'chapter_title': 'Agentic RAG'},
 {'chapter_num': '14', 'chapter_title': 'Agentic RAG', 'chapter_num_int': 14}]

## 8. The recall/precision trade-off: what filtering costs you

Filtering is not free. It guarantees a result *stays inside* the chapters you allow — but if the chunk you actually need lives in a chapter your filter excludes, no amount of embedding similarity will bring it back. Reusing notebook 5's rank-of-correct-chunk check, now with an optional `filter`:

In [11]:
hallucination_chunk_idx = next(i for i, d in enumerate(chunks) if "dynamic, external knowledge source" in d.page_content)
paraphrase_query = "How can giving a language model outside documents stop it from making things up?"
target_chapter = chunks[hallucination_chunk_idx].metadata["chapter_num"]

def rank_of(target_idx, query, k=10, filter=None):
    results = vectorstore.similarity_search_with_score(query, k=k, filter=filter)
    for rank, (doc, score) in enumerate(results, start=1):
        if doc.page_content == chunks[target_idx].page_content:
            return rank
    return f"> {k}"

print(f"Target chunk lives in chapter {target_chapter} ({chunks[hallucination_chunk_idx].metadata['chapter_title']})")
print("No filter:                       rank", rank_of(hallucination_chunk_idx, paraphrase_query))
print("Filtered to its own chapter:     rank", rank_of(hallucination_chunk_idx, paraphrase_query, filter={"chapter_num": target_chapter}))
print("Filtered to exclude its chapter: rank", rank_of(hallucination_chunk_idx, paraphrase_query, filter={"chapter_num": {"$ne": target_chapter}}))

Target chunk lives in chapter 01 (Introduction to RAG)
No filter:                       rank > 10
Filtered to its own chapter:     rank 7
Filtered to exclude its chapter: rank > 10


## Common pitfalls

- **Range operators need numeric metadata.** `$gt`/`$gte`/`$lt`/`$lte` raise `ValueError` on a string field, zero-padded or not — store a numeric copy (`chapter_num_int`) if you'll ever need ranges.
- **Only one top-level operator per `where` clause.** `{"a": 1, "b": 2}` is *not* an implicit AND — wrap multi-condition filters in `{"$and": [...]}` explicitly, or Chroma raises `ValueError`.
- **`filter={}` is not "no filter."** It hits the same "exactly one operator" rule and raises. To search unfiltered, omit the `filter` argument (or pass `filter=None`) — don't pass an empty dict expecting it to mean "everything."
- **A filter can filter out the right answer.** Scoping to the wrong chapter (or excluding the right one) makes a chunk unreachable regardless of how well it matches semantically — filtering trades recall for precision, and it's a hard cutoff, not a soft re-ranking.
- **A narrow filter shrinks the candidate pool `k` is drawn from.** If fewer chunks match your filter than `k`, you'll simply get fewer results back, not padding from outside the filter.

## Takeaways

- Chroma's `filter` argument maps onto a `where` clause evaluated before the similarity ranking runs, using `$eq`/`$ne`/`$in`/`$nin` for exact/set matching and `$gt`/`$gte`/`$lt`/`$lte` for numeric ranges — the latter requiring genuinely numeric metadata.
- `$and` / `$or` combine conditions, but Chroma allows only one top-level operator, so multi-condition filters must be explicit.
- The same `filter` works through `.as_retriever()`, so scoped search plugs into chains exactly like unscoped search did in notebook 5.
- Filtering is a hard boundary, not a soft preference — it can guarantee scope, but it can just as easily filter the right chunk out of reach.

**Next up (notebook 7):** turning a retriever — filtered or not — into a full RAG chain that feeds retrieved chunks to an LLM and generates a grounded answer.